# Fase 1 — Evaluación de geocodificación inversa y detección de carretera

**Proyecto:** Panel de viaje para coche (`tragabytes/panel-viaje`)
**Fase del plan:** 1.1 (geocodificación inversa) + 1.2 (carretera actual)

## Qué hace este notebook

Evalúa **dos APIs gratuitas de geocodificación inversa** (Nominatim y Photon) para el panel de viaje, y además comprueba si Nominatim sirve para detectar la **carretera actual** cuando se circula por una autovía.

El notebook se estructura en cuatro pruebas:

1. **Nominatim a zoom 14** sobre 12 coordenadas repartidas por España (una por CCAA + 2 casos especiales): comprueba que resuelve correctamente municipio, provincia y comunidad autónoma.
2. **Photon a zoom por defecto** sobre las mismas 12 coordenadas: comparativa directa frente a Nominatim.
3. **Comparativa lado a lado** de las dos APIs.
4. **Nominatim a zoom 17** sobre 7 puntos de autovías españolas: comprueba si devuelve el nombre/código de la carretera al pedir con zoom de calle.

Al final de cada prueba se muestra una tabla y un resumen con latencias medias y tamaños de respuesta.

## Cómo ejecutarlo

`Entorno de ejecución → Ejecutar todas`. Tarda alrededor de 30 segundos (el cuello de botella es la política de 1 petición/segundo de Nominatim).

## Importante sobre la latencia

La latencia medida aquí **no es la del móvil en el coche**. Colab corre en servidores de Google con fibra; el móvil irá por datos compartidos y verá latencias más altas. La medición real se hará en la fase 5. Lo que sí es fiable en Colab: qué campos devuelve cada API, calidad de los datos, comparativa entre ambas y tamaño de las respuestas.

## Resultados resumidos (ver fichas en `docs/seguimiento_desarrollo_panel_viaje.docx`)

- **Nominatim: apta.** 12/12 municipios y CCAA correctos, 7/7 autovías detectadas, latencia media 191 ms a zoom 14 y 149 ms a zoom 17 (desde Colab).
- **Photon: descartada como geocodificador administrativo.** Devuelve el POI más cercano en lugar del contenedor administrativo; no da provincia fiable (devuelve comarcas) y no detecta la carretera actual. Queda parqueada como posible fuente secundaria para POIs en la fase 1.4.
- **Detección de autovía:** Nominatim a zoom 17 devuelve el nombre de la vía en el campo `road`. Las autovías radiales (A-1 a A-6) vienen con nombre descriptivo ("Autovía del Nordeste") en lugar del código; las demás vienen con código directo. Se resolverá con una tabla estática de mapeo en el panel, sin peticiones adicionales.


## Preparación

Imports, constantes y lista de coordenadas de prueba.

In [ ]:
import requests
import time
import pandas as pd
from IPython.display import display

# Cabecera obligatoria para Nominatim y recomendada para Photon
HEADERS = {
    "User-Agent": "panel-viaje-test/0.1 (github.com/tragabytes/panel-viaje)"
}

# 10 coordenadas (una por CCAA) + 2 casos especiales del plan
# Usadas en las pruebas 1, 2 y 3.
COORDS = [
    # (nombre_esperado, ccaa_esperada, lat, lon, nota)
    ("Ronda",                    "Andalucía",          36.7428, -5.1658, "pueblo mediano"),
    ("Albarracín",               "Aragón",             40.4064, -1.4433, "pueblo pequeño histórico"),
    ("Cangas de Onís",           "Asturias",           43.3509, -5.1299, "pueblo pequeño"),
    ("Valldemossa",              "Islas Baleares",     39.7104,  2.6231, "pueblo pequeño"),
    ("Teguise",                  "Canarias",           29.0607,-13.5626, "pueblo pequeño"),
    ("Santillana del Mar",       "Cantabria",          43.3929, -4.1085, "pueblo pequeño turístico"),
    ("Almagro",                  "Castilla-La Mancha", 38.8898, -3.7125, "pueblo pequeño"),
    ("La Alberca",               "Castilla y León",    40.4919, -6.1122, "pueblo pequeño"),
    ("Besalú",                   "Cataluña",           42.1993,  2.6985, "pueblo pequeño"),
    ("Chinchón",                 "Madrid",             40.1394, -3.4266, "pueblo pequeño"),
    # Casos especiales del plan:
    ("A-2 cerca de Medinaceli",  "Castilla y León",    41.1710, -2.4300, "punto sobre autovía (a zoom 14 resuelve al municipio contenedor)"),
    ("Límite Valencia/Castellón","C. Valenciana",      39.7500, -0.2300, "zona límite provincial"),
]

# 7 coordenadas sobre firme de autovía, verificadas manualmente en OpenStreetMap.
# Usadas en la prueba 4.
AUTOVIA_COORDS = [
    ("A-2",   41.148078, -2.427866),
    ("A-1",   41.264409, -3.595552),
    ("A-6",   40.744217, -4.290319),
    ("A-5",   40.116511, -4.275809),
    ("A-4",   39.618136, -3.522518),
    ("AP-36", 39.632300, -3.080563),
    ("A-3",   39.872121, -2.608156),
]

print(f"Coordenadas generales: {len(COORDS)}")
print(f"Coordenadas de autovía: {len(AUTOVIA_COORDS)}")


## Prueba 1 — Nominatim a zoom 14

**Endpoint:** `https://nominatim.openstreetmap.org/reverse`
**Política:** máximo 1 petición/segundo, `User-Agent` obligatorio.
**Objetivo:** validar que devuelve municipio, provincia y CCAA correctos para las 12 coordenadas.

In [ ]:
NOMINATIM_URL = "https://nominatim.openstreetmap.org/reverse"

nominatim_rows = []
for name, ccaa, lat, lon, nota in COORDS:
    params = {
        "lat": lat, "lon": lon,
        "format": "json",
        "accept-language": "es",
        "zoom": 14,
        "addressdetails": 1,
    }
    try:
        t0 = time.time()
        r = requests.get(NOMINATIM_URL, params=params, headers=HEADERS, timeout=15)
        ms = (time.time() - t0) * 1000
        data = r.json()
        addr = data.get("address", {})

        muni = (addr.get("city") or addr.get("town") or addr.get("village")
                or addr.get("municipality") or addr.get("hamlet") or "—")
        prov = addr.get("province") or addr.get("county") or "—"
        state = addr.get("state") or "—"
        road = addr.get("road") or "—"

        nominatim_rows.append({
            "punto": name,
            "ccaa_esperada": ccaa,
            "muni": muni,
            "prov": prov,
            "state (CCAA)": state,
            "road": road,
            "ms": round(ms),
            "bytes": len(r.content),
        })
        print(f"✓ {name:28s} → {muni} / {prov} / {state}  [{round(ms)} ms]")
    except Exception as e:
        nominatim_rows.append({"punto": name, "ccaa_esperada": ccaa, "error": str(e)})
        print(f"✗ {name}: {e}")
    time.sleep(1.1)

df_nom = pd.DataFrame(nominatim_rows)
print("\n── Tabla Nominatim ──")
display(df_nom)


## Prueba 2 — Photon

**Endpoint:** `https://photon.komoot.io/reverse`
**Política:** requiere `User-Agent` (rechaza con 403 el de `requests` por defecto). Solo soporta los idiomas `default`, `de`, `en`, `fr`; se usa `default` para obtener los nombres en idioma local.
**Objetivo:** comparar calidad frente a Nominatim sobre las mismas 12 coordenadas.

In [ ]:
PHOTON_URL = "https://photon.komoot.io/reverse"

photon_rows = []
for name, ccaa, lat, lon, nota in COORDS:
    params = {"lat": lat, "lon": lon, "lang": "default"}
    try:
        t0 = time.time()
        r = requests.get(PHOTON_URL, params=params, headers=HEADERS, timeout=15)
        ms = (time.time() - t0) * 1000

        if r.status_code != 200:
            photon_rows.append({
                "punto": name, "ccaa_esperada": ccaa,
                "error": f"HTTP {r.status_code}: {r.text[:100]}"
            })
            print(f"✗ {name}: HTTP {r.status_code}")
            time.sleep(0.3)
            continue

        data = r.json()
        feats = data.get("features", [])

        if feats:
            p = feats[0].get("properties", {})
            muni  = (p.get("city") or p.get("town") or p.get("village")
                     or p.get("locality") or p.get("name") or "—")
            prov  = p.get("county") or "—"
            state = p.get("state") or "—"
            road  = p.get("street") or (p.get("name") if p.get("osm_key") == "highway" else "—")

            photon_rows.append({
                "punto": name, "ccaa_esperada": ccaa,
                "muni": muni, "prov": prov, "state (CCAA)": state,
                "road": road, "osm_key": p.get("osm_key"), "osm_value": p.get("osm_value"),
                "ms": round(ms), "bytes": len(r.content),
            })
            print(f"✓ {name:28s} → {muni} / {prov} / {state}  [{round(ms)} ms]")
        else:
            photon_rows.append({"punto": name, "ccaa_esperada": ccaa, "error": "sin resultados"})
            print(f"✗ {name}: sin resultados")
    except Exception as e:
        photon_rows.append({"punto": name, "ccaa_esperada": ccaa, "error": str(e)})
        print(f"✗ {name}: {e}")
    time.sleep(0.3)

df_pho = pd.DataFrame(photon_rows)
print("\n── Tabla Photon ──")
display(df_pho)


## Prueba 3 — Comparativa lado a lado

Mismo punto, las dos APIs, para ver qué devuelve cada una y contrastarlas de un vistazo.

In [ ]:
comp_rows = []
for (name, ccaa, lat, lon, nota), n, p in zip(COORDS, nominatim_rows, photon_rows):
    comp_rows.append({
        "punto": name,
        "esperado (CCAA)": ccaa,
        "NOM muni":  n.get("muni", "ERROR"),
        "NOM prov":  n.get("prov", "ERROR"),
        "NOM CCAA":  n.get("state (CCAA)", "ERROR"),
        "NOM ms":    n.get("ms", "—"),
        "PHO muni":  p.get("muni", "ERROR"),
        "PHO prov":  p.get("prov", "ERROR"),
        "PHO CCAA":  p.get("state (CCAA)", "ERROR"),
        "PHO ms":    p.get("ms", "—"),
    })

df_comp = pd.DataFrame(comp_rows)
pd.set_option("display.max_columns", None)
pd.set_option("display.width", 200)
display(df_comp)


def resumen(rows, label):
    ok = [r for r in rows if "error" not in r]
    if not ok:
        print(f"{label}: 0/{len(rows)} OK")
        return
    avg_ms = sum(r["ms"] for r in ok) / len(ok)
    avg_sz = sum(r["bytes"] for r in ok) / len(ok)
    print(f"── {label} ──")
    print(f"  Peticiones OK:  {len(ok)}/{len(rows)}")
    print(f"  Latencia media: {avg_ms:.0f} ms")
    print(f"  Tamaño medio:   {avg_sz:.0f} bytes")
    print()

print()
resumen(nominatim_rows, "NOMINATIM zoom 14")
resumen(photon_rows,    "PHOTON")


## Prueba 4 — Nominatim a zoom 17 sobre autovías

**Objetivo:** comprobar si Nominatim devuelve el nombre de la vía cuando se pide con zoom de calle sobre un punto en el firme de una autovía.

Las 7 coordenadas han sido verificadas manualmente en OpenStreetMap para garantizar que caen sobre el asfalto de la autovía esperada.

In [ ]:
autovia_rows = []
for expected_road, lat, lon in AUTOVIA_COORDS:
    params = {
        "lat": lat, "lon": lon,
        "format": "json",
        "accept-language": "es",
        "zoom": 17,
        "addressdetails": 1,
        "extratags": 1,
    }
    try:
        t0 = time.time()
        r = requests.get(NOMINATIM_URL, params=params, headers=HEADERS, timeout=15)
        ms = (time.time() - t0) * 1000
        data = r.json()
        addr = data.get("address", {})
        extra = data.get("extratags", {}) or {}

        road = addr.get("road") or "—"
        name_top = data.get("name") or "—"
        osm_class = data.get("class", "—")
        osm_type = data.get("type", "—")
        ref = extra.get("ref") or "—"

        # ¿El código esperado aparece en alguno de los campos?
        match = "✓" if (expected_road in str(ref)
                        or expected_road in str(road)
                        or expected_road in str(name_top)) else "?"

        autovia_rows.append({
            "esperado": expected_road,
            "road (address)": road,
            "name (top)": name_top,
            "ref (extratags)": ref,
            "class/type": f"{osm_class}/{osm_type}",
            "match": match,
            "ms": round(ms),
            "bytes": len(r.content),
        })
        print(f"{match} esperado={expected_road}  →  road={road}  (class={osm_class}/{osm_type})")
    except Exception as e:
        autovia_rows.append({"esperado": expected_road, "error": str(e)})
        print(f"✗ {expected_road}: {e}")
    time.sleep(1.1)

df_autovias = pd.DataFrame(autovia_rows)
print("\n── Tabla autovías ──")
display(df_autovias)

ok = [r for r in autovia_rows if "error" not in r]
matches = [r for r in ok if r["match"] == "✓"]
motorway = [r for r in ok if "motorway" in r["class/type"]]
avg_ms = sum(r["ms"] for r in ok) / len(ok) if ok else 0
print(f"\nClasificadas como highway/motorway: {len(motorway)}/{len(ok)}")
print(f"Código devuelto directamente (match exacto): {len(matches)}/{len(ok)}")
print(f"Latencia media zoom 17: {avg_ms:.0f} ms")
print()
print("Observación: los casos con '?' NO son errores. Nominatim detecta correctamente")
print("que estamos sobre autovía (class=highway/motorway), pero devuelve el nombre")
print("descriptivo (ej. 'Autovía del Nordeste') en lugar del código (A-2). Esto se")
print("resuelve en el panel con una tabla estática de mapeo nombre descriptivo → código.")


## Conclusiones

### Nominatim — apta, es la API principal
- 12/12 municipios correctos a zoom 14, incluidos pueblos pequeños.
- 12/12 CCAA correctas (en formato completo oficial: "Comunidad de Madrid", "Comunitat Valenciana", etc.).
- 7/7 detecta autovía a zoom 17 (`class=highway/motorway`).
- Latencia media 191 ms a zoom 14 y 149 ms a zoom 17 desde Colab.
- Tamaño medio 677 bytes (zoom 14) / 870 bytes (zoom 17). Despreciable.

**Limitaciones conocidas:**
- Para autovías radiales (A-1 a A-6) devuelve el nombre descriptivo ("Autovía del Nordeste") en lugar del código. Se resuelve con tabla estática de mapeo en cliente.
- El tag `ref` de OSM no se expone de forma consistente en `extratags`, aunque existe en los datos subyacentes.

### Photon — descartada como geocodificador administrativo
- Problema de fondo: resuelve al POI más cercano (panadería, restaurante, parque) en lugar del contenedor administrativo. Los campos administrativos (`city`, `state`) se extraen por contexto.
- El campo `county` (provincia) devuelve comarcas en lugar de provincias españolas (Serranía de Ronda, Garrotxa, la Plana Baixa...). No fiable como fuente de provincia.
- No detecta la carretera actual: el campo `street` es la calle del POI más cercano, no la vía por la que se circula.
- No soporta idioma español como parámetro (solo `default`, `de`, `en`, `fr`).
- Nomenclatura mixta en regiones bilingües (Catalunya, Illes Balears, Comunitat Valenciana).
- Latencia media 647 ms desde Colab (más del triple que Nominatim).

**Queda parqueada** como posible fuente secundaria para la fase 1.4 (POIs por pueblo), dado que su capacidad de devolver el POI más cercano puede ser útil cuando Wikipedia no tenga datos de un pueblo pequeño.
